In [ ]:
using Pkg
Pkg.activate("..")

In [ ]:
using bslLD,Plots, Statistics
bslLD.greet()

In [ ]:
grid =  bslLD.Grid([0.0,-6.0],[10.0,6.0],[0.02,0.1],0.05,2000,1)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = v^2* exp(-v^2 / 2) / sqrt(2*pi)

f = bslLD.Distribution(grid, 0.01,initFuncv=initFuncv);
heatmap(f.data[:,:]',c=:viridis)

In [ ]:
mean(f.data, dims=1)

In [ ]:
rho = bslLD.ScalarField(bslLD.compute_density!(f,grid))
plot(rho.data)
phi = bslLD.poisson(rho,grid)
e = bslLD.compute_e(phi,grid)

plot(rho.data)
mean((rho.data.-mean(rho.data)).^2)

In [ ]:
rho2 = []
fdiag = []


for i in grid.itime
    bslLD.advectX!(f,grid,grid.dt)
    rho = bslLD.ScalarField(bslLD.compute_density!(f,grid))
    phi = bslLD.poisson(rho,grid)
    e = bslLD.compute_e(phi,grid)
    bslLD.advectV!(f,grid,grid.dt,e)
    push!(rho2, mean((phi.data.-mean(phi.data)).^2))
    if i%10==0
        push!(fdiag,f.data[:,:])
    end
end
heatmap(f.data[:,:]',c=:viridis)


In [ ]:
plot(rho2, yscale=:log)

In [ ]:
num_frames = size(fdiag, 1) # Or length(fdiag) if it's a Vector of 2D arrays
frames_to_plot = 1:num_frames

animation = @animate for i in frames_to_plot
    heatmap(fdiag[i],
        title = "Frame $i",
        xlabel = "X-axis",
        ylabel = "Y-axis",
    )
end

gif(animation, "fdiag_heatmap_animation.gif", fps = 10) # fps is frames per second

